# Time Series Forecasting - Weather Data

This notebook solves the humidity forecasting task using **ARIMA** and **SARIMAX**.

It includes:
- data loading and preparation,
- stationarity checks,
- trend/seasonality handling,
- model training and evaluation,
- forecasting humidity for the next 100 days.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import mean_absolute_error, mean_squared_error
import math
import itertools

plt.style.use("default")

## 1) Load and prepare the dataset

In [ ]:
# Update the path if needed
DATA_PATH = "weather_data.csv"

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

# Make sure the index is daily
df = df.asfreq("D")

# Handle missing values if any
df = df.interpolate(method="time").ffill().bfill()

df.head()

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Date range:", df.index.min().date(), "to", df.index.max().date())
print("Missing values:\n", df.isna().sum())

## 2) Focus on Humidity

In [ ]:
humidity = df["humidity"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(humidity, label="Humidity")
ax.set_title("Humidity Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Humidity")
ax.legend()
plt.show()

In [ ]:
rolling_mean = humidity.rolling(window=7).mean()
rolling_std = humidity.rolling(window=7).std()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(humidity, label="Original")
ax.plot(rolling_mean, label="7-day Rolling Mean")
ax.plot(rolling_std, label="7-day Rolling Std")
ax.set_title("Rolling Mean and Rolling Std")
ax.set_xlabel("Date")
ax.legend()
plt.show()

## 3) Stationarity check

In [ ]:
def adf_test(series, title="Series"):
    result = adfuller(series.dropna(), autolag="AIC")
    labels = ["ADF Statistic", "p-value", "# Lags Used", "# Observations Used"]
    out = pd.Series(result[0:4], index=labels)
    for key, val in result[4].items():
        out[f"Critical Value ({key})"] = val
    print(f"ADF Test for {title}")
    print(out)
    print()

adf_test(humidity, "Original Humidity")
adf_test(humidity.diff().dropna(), "First Difference")
adf_test(humidity.diff(7).dropna(), "Seasonal Difference (lag=7)")

In [ ]:
# Seasonal decomposition to inspect trend/seasonality
decomp = seasonal_decompose(humidity, model="additive", period=7)

fig = decomp.plot()
fig.set_size_inches(12, 8)
plt.show()

## 4) Train-test split

In [ ]:
train_size = int(len(humidity) * 0.8)
train = humidity.iloc[:train_size]
test = humidity.iloc[train_size:]

print("Train length:", len(train))
print("Test length:", len(test))
print("Train end date:", train.index[-1].date())
print("Test start date:", test.index[0].date())

## 5) ARIMA model

In [ ]:
# Small grid search to pick a reasonable ARIMA model by validation RMSE
best_arima = None
best_score = float("inf")
best_order = None
arima_results = []

for order in itertools.product(range(0, 3), range(0, 2), range(0, 3)):
    if order == (0, 0, 0):
        continue
    try:
        model = ARIMA(train, order=order)
        fit = model.fit()
        pred = fit.forecast(steps=len(test))
        rmse = math.sqrt(mean_squared_error(test, pred))
        arima_results.append((order, fit.aic, rmse))
        if rmse < best_score:
            best_score = rmse
            best_arima = fit
            best_order = order
    except Exception:
        pass

arima_df = pd.DataFrame(arima_results, columns=["order", "aic", "rmse"]).sort_values("rmse")
arima_df.head(10)

In [ ]:
print("Best ARIMA order:", best_order)
arima_pred = best_arima.forecast(steps=len(test))

arima_rmse = math.sqrt(mean_squared_error(test, arima_pred))
arima_mae = mean_absolute_error(test, arima_pred)

print("ARIMA RMSE:", round(arima_rmse, 4))
print("ARIMA MAE :", round(arima_mae, 4))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train.index, train, label="Train")
ax.plot(test.index, test, label="Test")
ax.plot(test.index, arima_pred, label=f"ARIMA{best_order} Prediction")
ax.set_title("ARIMA Predictions on Test Set")
ax.set_xlabel("Date")
ax.set_ylabel("Humidity")
ax.legend()
plt.show()

## 6) SARIMAX model

In [ ]:
# Daily weather data usually shows short weekly patterns, so a 7-day seasonal period is used.
sarimax_order = (1, 1, 1)
sarimax_seasonal_order = (1, 1, 1, 7)

sarimax_model = SARIMAX(
    train,
    order=sarimax_order,
    seasonal_order=sarimax_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarimax_fit = sarimax_model.fit(disp=False)
sarimax_pred = sarimax_fit.forecast(steps=len(test))

sarimax_rmse = math.sqrt(mean_squared_error(test, sarimax_pred))
sarimax_mae = mean_absolute_error(test, sarimax_pred)

print("SARIMAX order:", sarimax_order)
print("SARIMAX seasonal order:", sarimax_seasonal_order)
print("SARIMAX RMSE:", round(sarimax_rmse, 4))
print("SARIMAX MAE :", round(sarimax_mae, 4))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train.index, train, label="Train")
ax.plot(test.index, test, label="Test")
ax.plot(test.index, sarimax_pred, label="SARIMAX Prediction")
ax.set_title("SARIMAX Predictions on Test Set")
ax.set_xlabel("Date")
ax.set_ylabel("Humidity")
ax.legend()
plt.show()

## 7) Compare models

In [ ]:
comparison = pd.DataFrame({
    "Model": ["ARIMA", "SARIMAX"],
    "RMSE": [arima_rmse, sarimax_rmse],
    "MAE": [arima_mae, sarimax_mae]
}).sort_values("RMSE")

comparison

## 8) Forecast humidity for the next 100 days with the best model

In [ ]:
# Fit the better model on the full dataset and forecast 100 future days
best_model_name = comparison.iloc[0]["Model"]

if best_model_name == "SARIMAX":
    final_model = SARIMAX(
        humidity,
        order=sarimax_order,
        seasonal_order=sarimax_seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)
else:
    final_model = ARIMA(humidity, order=best_order).fit()

future_steps = 100
future_forecast = final_model.forecast(steps=future_steps)

future_dates = pd.date_range(start=humidity.index[-1] + pd.Timedelta(days=1), periods=future_steps, freq="D")
forecast_df = pd.DataFrame({
    "Date": future_dates,
    "Forecasted Humidity": future_forecast.values
})

forecast_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(humidity.index, humidity, label="Historical Humidity")
ax.plot(forecast_df["Date"], forecast_df["Forecasted Humidity"], label="100-Day Forecast")
ax.set_title("Humidity Forecast for the Next 100 Days")
ax.set_xlabel("Date")
ax.set_ylabel("Humidity")
ax.legend()
plt.show()

In [ ]:
forecast_df.tail(10)

## 9) Conclusion

- The humidity series was checked for stationarity using the ADF test.
- Trend and seasonality were inspected using rolling statistics and seasonal decomposition.
- ARIMA and SARIMAX were trained and evaluated on a hold-out test set.
- The better model was then used to forecast humidity for the next 100 days.